# **Exercise 09: *Plotly***

Prepare the project:

In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas
%pip install plotly

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Import allowed modules:

In [3]:
import sqlite3

import pandas as pd
import plotly.graph_objects as go

Import allowed entities:

In [4]:
from pandas import DataFrame
from sqlite3 import Connection
from plotly.graph_objects import Figure

Create a dictionary for connection to the database:

In [5]:
conn_params: dict = {"db_file": "checking_logs.sqlite", "db_file_path": "../../data/", }

Make a connection to the [database](https://drive.google.com/open?id=1zQ8AR2Ry3ajzB3UZO1Sfk3xtDJlzQF2M):

In [6]:
conn: Connection = sqlite3.connect(conn_params["db_file_path"] + conn_params["db_file"], )

Check connection:

In [7]:
conn.cursor().execute("SELECT 123;", );

Create a query:

In [8]:
sql_q: str = """
             SELECT
                 *
             FROM
                 checker
             WHERE
                 checker.uid LIKE 'user_%';
             """

Run a query:

In [9]:
df: DataFrame = pd.io.sql.read_sql(
    sql_q,
    conn,
    parse_dates=["timestamp", ],
)

Chech a result:

In [10]:
df.head()

,index,status,success,timestamp,numTrials,labname,uid
0,7,ready,0,2020-04-17 05:19:02.744528,1,project1,user_4
1,8,checking,0,2020-04-17 05:22:35.249331,2,project1,user_4
2,9,ready,1,2020-04-17 05:22:45.549397,2,project1,user_4
3,10,checking,1,2020-04-17 05:34:14.691200,3,project1,user_4
4,11,ready,1,2020-04-17 05:34:24.422370,3,project1,user_4


Take into account only logs from the table `checker` where the `status` is `ready`:

In [11]:
df = df[(df["status"] == "ready") & (df["labname"] == "project1")]

Check updated *Pandas* dataframe:

In [12]:
df.head()

,index,status,success,timestamp,numTrials,labname,uid
0,7,ready,0,2020-04-17 05:19:02.744528,1,project1,user_4
2,9,ready,1,2020-04-17 05:22:45.549397,2,project1,user_4
4,11,ready,1,2020-04-17 05:34:24.422370,3,project1,user_4
6,13,ready,0,2020-04-17 05:43:27.773992,4,project1,user_4
8,15,ready,1,2020-04-17 05:46:32.275104,5,project1,user_4


Create a list of useless *Pandas* dataframe columns:

In [13]:
useless_cols: list = [
    "index",
    "status",
    "success",
    "labname",
]

Drop them:

In [14]:
df.drop(columns=useless_cols, inplace=True, )

Check updated *Pandas* dataframe:

In [15]:
df.head()

,timestamp,numTrials,uid
0,2020-04-17 05:19:02.744528,1,user_4
2,2020-04-17 05:22:45.549397,2,user_4
4,2020-04-17 05:34:24.422370,3,user_4
6,2020-04-17 05:43:27.773992,4,user_4
8,2020-04-17 05:46:32.275104,5,user_4


Create a temporary lists:

In [16]:
users: list = df["uid"].unique()
tss: list = df["timestamp"].unique()

Create initial plots:

In [17]:
init_plots: list = []

for user in users:
    init_plots.append(go.Scatter(
        x=df[(df["uid"] == user) & (df["timestamp"] == tss[0])]["timestamp"],
        y=df[(df["uid"] == user) & (df["timestamp"] == tss[0])]["numTrials"],
        mode="lines+markers",
        name=user,
    ), )

Create a list of an epochs for visualization:

In [18]:
epochs: list = []

for ts in tss:
    traces: list = []
    user_epoch: DataFrame = df[df["timestamp"] < ts]

    for user in users:
        traces.append(go.Scatter(
            x=user_epoch[user_epoch["uid"] == user]["timestamp"],
            y=user_epoch[user_epoch["uid"] == user]["numTrials"],
            mode="lines+markers",
            name=user,
        ), )

    epochs.append(go.Frame(data=traces, name=str(ts, ), ), )

Create a race plot entity:

In [19]:
fig: Figure = go.Figure(
    data=init_plots,
    layout=go.Layout(
        title = "Dynamic of commits per user in project1",
        xaxis = dict(title="Timestamp", ),
        yaxis = dict(title="Number of trials", ),
        updatemenus=[{
            "buttons":[
                {
                    "args": [None, { "frame": {"duration": 350, "redraw": True, }, "fromcurrent": True, }, ],
                    "label": "Play",
                    "method": "animate",
                },
                {
                    "args": [[None, ], {
                        "frame": {"duration": 0, "redraw": True, },
                        "mode": "immediate",
                        "transition": {"duration": 0, },
                        }, ],
                    "label": "Pause",
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {'r': 10, 't': 87, },
            "showactive": False,
            "type": "buttons",
            'x': 0.1,
            "xanchor": "right",
            'y': 0,
            "yanchor": "top",
        }, ],
    ),
    frames=epochs,
)

Draw it:

In [20]:
fig.show()

Close the connection to the database:

In [21]:
conn.close()